# Attention explorer: IO gap-fillers, one day at a time

The five IO attention gap-fillers and the Conv3d reference on 12 preset val and 12 preset test days:
`io2_tattn8` (per-pixel attention, `chl_cnn/tattn.py`), `io3_ptok4` and `io3_ptok4_lavg` (patch tokens with a per-pixel day blend, `chl_cnn/ptok.py`),
`io4_tokdec` and `io4_tokdec_blend` (token → decoder stem, `chl_cnn/tokdec.py`) and `t3d_dil_c4_60ep_gdl100` (Conv3d stem, ±4 days).
Schematics: `analyses/io3_attn/figures/ptok_schematic.png`, `analyses/io4_tokdec/figures/tokdec_schematic.png`.

The notebook reads only the caches in `analyses/attn_viz/explorer_cache/<run>.npz` (built on CPU by `analyses/attn_viz/build_explorer.py`
with the trainer's own input path and the stem recorders; 12 days x 2 splits, about 12 min per run plus the store load, up to 15 min when the store is not in the page cache; ~14 GB RAM). A run without a cache prints its build command below.
Preset days (`analyses/attn_viz/explorer_days.json`): per split, one day per season × observed-fraction tercile, all with ≥ 500 hidden pixels;
val includes the io3/io4 weight-map dates 2001-03-02, 2001-10-29 and 2007-06-27. Hidden pixels are the trainer's fixed fake-cloud masks
(observed on day t, not visible in the input). Maps are north up, land blank. Plot code: `chl_cnn/attn_explorer.py`.

**Controls:** *model*, an optional second model (*compare with*), the split and the preset day. All panels follow them.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
%matplotlib inline
import matplotlib.pyplot as plt

plt.ioff()  # figures go only where display() puts them (inside the Output widgets)
from chl_cnn import attn_explorer as AE

ex = AE.Explorer(a="io4_tokdec_blend", b=None)
ex.controls()

## (a) Reconstruction

Day t's visible ln CHL (the model's input on day t), the prediction, the truth on the hidden pixels and the error there (prediction − truth).
The row title gives the day's hidden-pixel MAE (ln CHL) and n. With a second model, one row per model on shared colour scales.

In [ ]:
ex.recon()

## (b) Attention

Weight maps by day offset (columns; pick them in *offsets*), each next to that day's visible mask in the model's input (donor-masked).
Weight sets per stem: `attn` per-pixel pooling weights over the window days, day t included (mean over heads); `patch` per-pixel day weights
over the context days visible at the pixel plus a "no day" slot, with the two logit terms (patch score interpolated between patch centres,
per-pixel term; `lavg` also the local confidence conv(m)); `tokdec` the pooling weight of each 4 × 4 px patch (day t included, mean over heads,
shown as blocks), and for `tokdec_blend` also the blend day weights with their two logit terms. Weight colour scales run from 0 to the
99.5th percentile of the shown columns. The bar chart gives the day's mean weight per hidden pixel by offset, split by whether the day is
visible at the pixel, and the "no day" share; the bars of a panel sum to 1. The Conv3d reference has no weights.

In [ ]:
ex.attention()

## (c) Attribution

Integrated gradients of the mean predicted ln CHL over day t's hidden pixels with respect to the input groups: context-day CHL (all t ≠ 0),
day-t visible CHL, SST, salinity, MLD, SLA, currents (u, v), winds (u, v), air temperature (each over every window day), and the static
channels (climatology, coordinates, depth, day of year, year; for patch and token-decoder stems also the per-day climatology). Baseline: all
attributed inputs at 0 (the training mean in z-space); the visible-mask channels stay at their real values. 32 Gauss-Legendre nodes.
Left: each group's share of the day's total |attribution|. Right: the |attribution| map of the group picked in *map group*, summed over the
group's channels and window days, on a log colour scale from 0.001 × to 1 × its 99.5th percentile over ocean pixels. The title gives
f(x) − f(baseline) and the completeness error (sum of attributions minus that difference).

In [ ]:
ex.attribution()

## (d) Context

SST on day t (°C) with the surface-current vectors (every 6th pixel each way) and the hidden pixels outlined, for the first model's day
(the fields are the same for every model).

In [ ]:
ex.context()